# Euclid straight-line artifact detection - version 3.2

This Colab notebook detects clear, long, straight image artifacts in the
Euclid morphology JPEG cutouts. It deliberately ignores saturation, noise,
halos, curved structures, and generic elongation.

Version 3.2 retains broad-trail detection while reducing false exclusions:

- every image is rescaled to a common analysis size before line detection;
- probabilistic Hough segments recover locally continuous trails;
- global Hough exclusion lines are proposed from edges; bright-residual
  proposals can optionally enrich the review-only list;
- each global line is evaluated with several line widths to recover thin and
  broad artifacts;
- strict global Hough lines must come from edges and remain bright across
  their entire length; bright-residual proposals remain review-only;
- visually line-free and edge-on-galaxy cutouts are negative regressions.

The expensive stages are lazy: the residual Hough transform runs only when the
edge transform finds no strict artifact and review-only scanning is enabled,
additional widths are evaluated only
for promising peaks, and an image stops as soon as a strict line is found.

The notebook never deletes source images. It creates metrics, review, and
strict exclusion manifests. Labelled A/B/C lens candidates are retained in the
filtered catalogue and surfaced for manual review.

In [ ]:
import subprocess
import sys

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "opencv-python-headless>=4.9,<5",
    "pandas>=2.0",
    "pyarrow>=14",
    "Pillow>=10",
    "tqdm>=4.66",
    "matplotlib>=3.8",
])


In [ ]:
from google.colab import drive

drive.mount("/content/drive")


## Configuration

Begin with a representative random sample. Set `MAX_IMAGES = None` only after
reviewing the diagnostic gallery. Checkpoints are versioned, so a stopped Colab
session can resume safely.


In [ ]:
from pathlib import Path

MORPH_PATH = Path(
    "/content/drive/MyDrive/catalogues/morphology_catalogue/"
    "morphology_catalogue.parquet"
)
CUTOUT_BASE = Path(
    "/content/drive/MyDrive/catalogues/morphology_catalogue/"
    "cutouts_jpg_gz_arcsinh_vis_only"
)
LENS_PATH = Path(
    "/content/drive/MyDrive/catalogues/strong_lensing_catalogue/"
    "q1_discovery_engine_lens_catalog.csv"
)
OUTPUT_DIR = Path(
    "/content/drive/MyDrive/catalogues/morphology_catalogue/"
    "straight_line_artifact_detection"
)

DETECTOR_VERSION = "v3_2_continuous_multiscale_hough_lines"

# None processes the complete catalogue. Use a smaller integer while tuning.
MAX_IMAGES = None
RANDOM_SAMPLE = False
RANDOM_SEED = 42
BATCH_SIZE = 1_000
N_WORKERS = 4  # Fallback when automatic tuning is disabled.
AUTO_TUNE_WORKERS = True
WORKER_OPTIONS = (2, 4, 8, 12)
WORKER_BENCHMARK_IMAGES = 64  # Distinct images per option.
INCLUDE_BRIGHT_REVIEW = False  # Does not affect strict exclusions.

# A common scale makes thresholds comparable across differently sized cutouts.
DETECTION_SIZE = 256

# Probabilistic Hough candidate and strict thresholds.
MIN_HOUGH_LENGTH_RATIO = 0.45
MIN_LINE_BRIGHT_SUPPORT = 0.30
MIN_LINE_CONTRAST = 1.10
STRICT_LINE_LENGTH_RATIO = 0.65
STRICT_SINGLE_BORDER_LENGTH_RATIO = 0.82

# Global Hough proposals from edges and bright residual pixels.
GLOBAL_EDGE_HOUGH_VOTE_RATIO = 0.16
GLOBAL_BRIGHT_HOUGH_VOTE_RATIO = 0.20
GLOBAL_HOUGH_MIN_VOTES = 18
GLOBAL_MAX_HOUGH_PEAKS = 64
GLOBAL_CORE_WIDTH_RATIOS = (0.020, 0.040, 0.008, 0.070)
GLOBAL_PROMISING_MIN_BRIGHT_SUPPORT = 0.35
GLOBAL_PROMISING_MIN_EDGE_SUPPORT = 0.70

# Review candidates remain intentionally permissive.
GLOBAL_REVIEW_MIN_CHORD_RATIO = 0.68
GLOBAL_REVIEW_MIN_BRIGHT_SUPPORT = 0.30
GLOBAL_REVIEW_MIN_CONTRAST = 1.15
GLOBAL_REVIEW_MIN_EDGE_SUPPORT = 0.45

# Conservative strict rule retained from version 2.
GLOBAL_STRICT_MIN_CHORD_RATIO = 0.70
GLOBAL_STRICT_MIN_BRIGHT_SUPPORT = 0.45
GLOBAL_STRICT_MIN_CONTRAST = 1.30
GLOBAL_STRICT_MIN_EDGE_SUPPORT = 0.50

# Version 3 strict rule: slightly lower intensity thresholds require much
# stronger edge coherence. This recovers broad trails without accepting the
# tested A/B/C lens examples.
COHERENT_STRICT_MIN_CHORD_RATIO = 0.70
COHERENT_STRICT_MIN_BRIGHT_SUPPORT = 0.40
COHERENT_STRICT_MIN_CONTRAST = 1.22
COHERENT_STRICT_MIN_EDGE_SUPPORT = 0.85

# Automatic global exclusion requires all 12 equal-length segments to
# have bright support in at least 25% of their core pixels.
GLOBAL_CONTINUITY_BINS = 12
GLOBAL_MIN_BRIGHT_FRACTION_PER_BIN = 0.25
GLOBAL_STRICT_MIN_SUPPORTED_BINS = 12
GLOBAL_STRICT_SOURCE = "edges"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PARTS_DIR = OUTPUT_DIR / f"parts_{DETECTOR_VERSION}"
PARTS_DIR.mkdir(parents=True, exist_ok=True)

assert MORPH_PATH.exists(), MORPH_PATH
assert CUTOUT_BASE.exists(), CUTOUT_BASE

## Build the cutout manifest

Paths are derived from `morphology_catalogue.parquet`; the notebook does not
recursively list hundreds of thousands of Google Drive files.


In [ ]:
import pandas as pd


def normalize_object_id(value) -> str:
    text = str(value).strip().strip("'\"")
    return text[:-2] if text.endswith(".0") else text


def filename_object_id(value) -> str:
    object_id = normalize_object_id(value)
    if object_id.startswith("-"):
        return f"NEG{object_id[1:]}"
    return object_id.replace("-", "NEG")


def build_cutout_path(row) -> str:
    parts = str(row.id_str).strip().split("_")
    if len(parts) == 2:
        tile, id_from_name = parts
    elif len(parts) >= 3:
        tile, id_from_name = parts[-2], parts[-1]
    else:
        raise ValueError(f"Cannot parse id_str={row.id_str!r}")
    file_id = filename_object_id(id_from_name or row.object_id)
    filename = f"{tile}_{file_id}_gz_arcsinh_vis_only.jpg"
    return str(CUTOUT_BASE / tile / filename)


catalogue = pd.read_parquet(MORPH_PATH, columns=["object_id", "id_str"])
catalogue["object_id"] = catalogue["object_id"].map(normalize_object_id)
catalogue = catalogue.dropna(subset=["id_str"]).drop_duplicates("object_id")

if MAX_IMAGES is not None:
    if RANDOM_SAMPLE:
        catalogue = catalogue.sample(
            n=min(MAX_IMAGES, len(catalogue)),
            random_state=RANDOM_SEED,
        )
    else:
        catalogue = catalogue.head(MAX_IMAGES)

manifest = catalogue.copy()
manifest["image_path"] = [
    build_cutout_path(row) for row in manifest.itertuples(index=False)
]

if LENS_PATH.exists():
    lens_df = pd.read_csv(LENS_PATH, dtype={"object_id": "string"})
    lens_df["object_id"] = lens_df["object_id"].map(normalize_object_id)
    if "grade" in lens_df.columns:
        lens_df["grade"] = lens_df["grade"].astype(str).str.upper().str.strip()
        lens_df = lens_df[lens_df["grade"].isin(["A", "B", "C"])]
    else:
        lens_df["grade"] = pd.NA
    lens_df = lens_df[["object_id", "grade"]].drop_duplicates("object_id")
    manifest = manifest.merge(lens_df, on="object_id", how="left")
else:
    manifest["grade"] = pd.NA

manifest["is_lens_candidate"] = manifest["grade"].isin(["A", "B", "C"])
manifest.reset_index(drop=True, inplace=True)
print(f"Objects in manifest: {len(manifest):,}")
manifest.head()


## Straight-line detector

The image is robustly normalized and a smooth background model is removed.
Canny and probabilistic Hough first propose locally continuous segments. Each
proposal is tested against the residual signal in a narrow core and adjacent
side bands.

A second, stricter standard Hough transform proposes full image lines. Each
line is clipped to the image boundaries and validated along the entire visible
chord using its length, residual contrast, bright-pixel support, and Canny-edge
continuity. This recovers obvious trails fragmented by noise, stars, galaxies,
or masks without broadly lowering the local detector thresholds.

In [ ]:
import math
import time

import cv2
import numpy as np
from PIL import Image

cv2.setNumThreads(1)


def robust_uint8(gray: np.ndarray) -> np.ndarray:
    gray = gray.astype(np.float32)
    p1, p99 = np.percentile(gray, (1, 99))
    if p99 <= p1:
        return np.zeros_like(gray, dtype=np.uint8)
    scaled = np.clip((gray - p1) / (p99 - p1), 0, 1)
    return np.round(scaled * 255).astype(np.uint8)


def prepare_detection_rgb(rgb: np.ndarray) -> np.ndarray:
    height, width = rgb.shape[:2]
    interpolation = (
        cv2.INTER_AREA
        if max(height, width) > DETECTION_SIZE
        else cv2.INTER_CUBIC
    )
    return cv2.resize(
        rgb,
        (DETECTION_SIZE, DETECTION_SIZE),
        interpolation=interpolation,
    )


def endpoint_near_border(
    x: float,
    y: float,
    width: int,
    height: int,
) -> bool:
    margin = max(2, int(round(min(width, height) * 0.04)))
    return (
        x <= margin
        or x >= width - 1 - margin
        or y <= margin
        or y >= height - 1 - margin
    )


def preprocess_for_lines(
    rgb: np.ndarray,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    analysis_rgb = prepare_detection_rgb(rgb)
    gray = cv2.cvtColor(analysis_rgb, cv2.COLOR_RGB2GRAY)
    normalized = robust_uint8(gray)

    background_size = max(15, int(round(DETECTION_SIZE * 0.24)))
    if background_size % 2 == 0:
        background_size += 1
    background = cv2.GaussianBlur(
        normalized,
        (background_size, background_size),
        0,
    )
    residual = cv2.absdiff(normalized, background)
    line_input = cv2.GaussianBlur(residual, (3, 3), 0)

    canny_low = max(16, int(np.percentile(line_input, 80)))
    canny_high = max(canny_low + 16, int(np.percentile(line_input, 96)))
    edges = cv2.Canny(
        line_input,
        canny_low,
        canny_high,
        apertureSize=3,
        L2gradient=True,
    )
    return analysis_rgb, line_input, edges


def detect_supported_lines(
    line_input: np.ndarray,
    edges: np.ndarray,
) -> tuple[int, list[dict[str, float]]]:
    height, width = edges.shape
    diagonal = math.hypot(width, height)

    lines = cv2.HoughLinesP(
        edges,
        rho=1,
        theta=np.pi / 180,
        threshold=max(20, int(round(min(width, height) * 0.35))),
        minLineLength=max(8, int(round(diagonal * MIN_HOUGH_LENGTH_RATIO))),
        maxLineGap=max(2, int(round(diagonal * 0.03))),
    )
    if lines is None:
        return 0, []

    high_response = float(np.percentile(line_input, 95))
    supported = []

    for x1, y1, x2, y2 in lines[:, 0, :]:
        length = math.hypot(x2 - x1, y2 - y1)
        length_ratio = length / diagonal
        core_width = max(3, int(round(min(width, height) * 0.006)))
        if core_width % 2 == 0:
            core_width += 1
        side_width = core_width + max(
            8,
            int(round(min(width, height) * 0.015)),
        )

        core_mask = np.zeros_like(edges, dtype=np.uint8)
        wide_mask = np.zeros_like(edges, dtype=np.uint8)
        cv2.line(core_mask, (x1, y1), (x2, y2), 255, core_width)
        cv2.line(wide_mask, (x1, y1), (x2, y2), 255, side_width)
        core_pixels = core_mask > 0
        side_pixels = (wide_mask > 0) & ~core_pixels
        if not np.any(core_pixels) or not np.any(side_pixels):
            continue

        core_values = line_input[core_pixels]
        core_mean = float(np.mean(core_values))
        side_mean = float(np.mean(line_input[side_pixels]))
        contrast = (core_mean + 1.0) / (side_mean + 1.0)
        bright_support = float(np.mean(core_values >= high_response))
        if (
            bright_support < MIN_LINE_BRIGHT_SUPPORT
            or contrast < MIN_LINE_CONTRAST
        ):
            continue

        border_endpoints = int(endpoint_near_border(x1, y1, width, height))
        border_endpoints += int(endpoint_near_border(x2, y2, width, height))
        clear_artifact = bool(
            length_ratio >= STRICT_LINE_LENGTH_RATIO
            and (
                border_endpoints == 2
                or (
                    border_endpoints >= 1
                    and length_ratio >= STRICT_SINGLE_BORDER_LENGTH_RATIO
                )
            )
        )
        supported.append({
            "x1": float(x1),
            "y1": float(y1),
            "x2": float(x2),
            "y2": float(y2),
            "length_ratio": float(length_ratio),
            "bright_support": bright_support,
            "contrast": contrast,
            "border_endpoints": float(border_endpoints),
            "clear_artifact": clear_artifact,
        })

    return int(len(lines)), supported


def clipped_hough_line(
    rho: float,
    theta: float,
    width: int,
    height: int,
) -> tuple[tuple[int, int], tuple[int, int]] | None:
    cos_theta = math.cos(theta)
    sin_theta = math.sin(theta)
    x0 = cos_theta * rho
    y0 = sin_theta * rho
    scale = 2 * (width + height)
    point_1 = (
        int(round(x0 - scale * sin_theta)),
        int(round(y0 + scale * cos_theta)),
    )
    point_2 = (
        int(round(x0 + scale * sin_theta)),
        int(round(y0 - scale * cos_theta)),
    )
    visible, clipped_1, clipped_2 = cv2.clipLine(
        (0, 0, width, height),
        point_1,
        point_2,
    )
    return (clipped_1, clipped_2) if visible else None


def global_line_artifact_score(line: dict[str, float]) -> float:
    chord_score = min(1.0, line["chord_ratio"] / 0.85)
    support_score = min(1.0, line["bright_support"] / 0.75)
    contrast_score = min(
        1.0,
        max(0.0, line["contrast"] - 1.0) / 0.50,
    )
    edge_score = min(1.0, line["edge_support"] / 0.90)
    return float(
        0.25 * chord_score
        + 0.30 * support_score
        + 0.25 * contrast_score
        + 0.20 * edge_score
    )


def line_band_metrics(
    line_input: np.ndarray,
    edge_neighborhood: np.ndarray,
    endpoints: tuple[tuple[int, int], tuple[int, int]],
    core_width_ratio: float,
    high_response: float,
) -> dict[str, float] | None:
    height, width = line_input.shape
    (x1, y1), (x2, y2) = endpoints
    core_width = max(1, int(round(min(width, height) * core_width_ratio)))
    if core_width % 2 == 0:
        core_width += 1
    side_width = core_width + max(
        8,
        int(round(min(width, height) * 0.035)),
    )

    core_mask = np.zeros_like(line_input, dtype=np.uint8)
    wide_mask = np.zeros_like(line_input, dtype=np.uint8)
    cv2.line(core_mask, (x1, y1), (x2, y2), 255, core_width)
    cv2.line(wide_mask, (x1, y1), (x2, y2), 255, side_width)
    core_pixels = core_mask > 0
    side_pixels = (wide_mask > 0) & ~core_pixels
    if not np.any(core_pixels) or not np.any(side_pixels):
        return None

    core_values = line_input[core_pixels]
    core_mean = float(np.mean(core_values))
    side_mean = float(np.mean(line_input[side_pixels]))
    return {
        "core_width": float(core_width),
        "bright_support": float(np.mean(core_values >= high_response)),
        "contrast": float((core_mean + 1.0) / (side_mean + 1.0)),
        "edge_support": float(np.mean(edge_neighborhood[core_pixels] > 0)),
    }


def count_supported_bright_bins(
    line_input: np.ndarray,
    endpoints: tuple[tuple[int, int], tuple[int, int]],
    core_width: int,
    high_response: float,
) -> int:
    (x1, y1), (x2, y2) = endpoints
    core_mask = np.zeros_like(line_input, dtype=np.uint8)
    cv2.line(core_mask, (x1, y1), (x2, y2), 255, core_width)
    ys, xs = np.nonzero(core_mask)
    if not len(xs):
        return 0
    dx, dy = x2 - x1, y2 - y1
    length_squared = max(dx * dx + dy * dy, 1)
    along = ((xs - x1) * dx + (ys - y1) * dy) / length_squared
    bin_index = np.clip(
        (along * GLOBAL_CONTINUITY_BINS).astype(int),
        0, GLOBAL_CONTINUITY_BINS - 1,
    )
    totals = np.bincount(bin_index, minlength=GLOBAL_CONTINUITY_BINS)
    bright = line_input[ys, xs] >= high_response
    bright_totals = np.bincount(
        bin_index, weights=bright, minlength=GLOBAL_CONTINUITY_BINS,
    )
    fractions = np.divide(
        bright_totals, totals,
        out=np.zeros(GLOBAL_CONTINUITY_BINS, dtype=float),
        where=totals > 0,
    )
    return int(np.count_nonzero(
        fractions >= GLOBAL_MIN_BRIGHT_FRACTION_PER_BIN
    ))


def hough_peaks(
    source_name: str,
    source: np.ndarray,
    vote_ratio: float,
) -> list[tuple[str, float, float]]:
    vote_threshold = max(
        GLOBAL_HOUGH_MIN_VOTES,
        int(round(min(source.shape) * vote_ratio)),
    )
    lines = cv2.HoughLines(
        source,
        rho=1,
        theta=np.pi / 360,
        threshold=vote_threshold,
    )
    if lines is None:
        return []
    return [
        (source_name, float(rho), float(theta))
        for rho, theta in lines[:GLOBAL_MAX_HOUGH_PEAKS, 0, :]
    ]


def global_peak_groups(
    line_input: np.ndarray,
    edges: np.ndarray,
    include_bright_review: bool,
) -> object:
    # This is a generator on purpose: the residual Hough transform is only
    # computed if no strict artifact is found in the edge peaks.
    yield hough_peaks(
        "edges",
        edges,
        GLOBAL_EDGE_HOUGH_VOTE_RATIO,
    )

    if not include_bright_review:
        return

    bright_mask = (
        line_input >= np.percentile(line_input, 90)
    ).astype(np.uint8) * 255
    bright_mask = cv2.morphologyEx(
        bright_mask,
        cv2.MORPH_CLOSE,
        np.ones((3, 3), dtype=np.uint8),
        iterations=1,
    )
    yield hough_peaks(
        "bright_residual",
        bright_mask,
        GLOBAL_BRIGHT_HOUGH_VOTE_RATIO,
    )


def classify_global_metrics(
    chord_ratio: float,
    metrics: dict[str, float],
    require_continuity: bool = True,
) -> tuple[bool, bool, bool]:
    contrast = metrics["contrast"]
    bright_support = metrics["bright_support"]
    edge_support = metrics["edge_support"]
    review_artifact = bool(
        contrast >= GLOBAL_REVIEW_MIN_CONTRAST
        and bright_support >= GLOBAL_REVIEW_MIN_BRIGHT_SUPPORT
        and edge_support >= GLOBAL_REVIEW_MIN_EDGE_SUPPORT
    )
    conservative_strict = bool(
        chord_ratio >= GLOBAL_STRICT_MIN_CHORD_RATIO
        and contrast >= GLOBAL_STRICT_MIN_CONTRAST
        and bright_support >= GLOBAL_STRICT_MIN_BRIGHT_SUPPORT
        and edge_support >= GLOBAL_STRICT_MIN_EDGE_SUPPORT
    )
    coherent_strict = bool(
        chord_ratio >= COHERENT_STRICT_MIN_CHORD_RATIO
        and contrast >= COHERENT_STRICT_MIN_CONTRAST
        and bright_support >= COHERENT_STRICT_MIN_BRIGHT_SUPPORT
        and edge_support >= COHERENT_STRICT_MIN_EDGE_SUPPORT
    )
    if require_continuity:
        continuous = (
            metrics.get("supported_bright_bins", 0)
            >= GLOBAL_STRICT_MIN_SUPPORTED_BINS
        )
        conservative_strict = conservative_strict and continuous
        coherent_strict = coherent_strict and continuous
    return review_artifact, conservative_strict, coherent_strict


def validate_global_continuity(
    line_input: np.ndarray,
    endpoints: tuple[tuple[int, int], tuple[int, int]],
    high_response: float,
    chord_ratio: float,
    metrics: dict[str, float],
) -> bool:
    _, conservative, coherent = classify_global_metrics(
        chord_ratio, metrics, require_continuity=False
    )
    if not (conservative or coherent):
        return False
    metrics["supported_bright_bins"] = count_supported_bright_bins(
        line_input, endpoints, int(metrics["core_width"]), high_response
    )
    _, conservative, coherent = classify_global_metrics(chord_ratio, metrics)
    return conservative or coherent


def metrics_quality(metrics: dict[str, float]) -> float:
    return float(
        0.35 * metrics["bright_support"]
        + 0.35 * min(max(metrics["contrast"] - 1.0, 0.0), 1.0)
        + 0.30 * metrics["edge_support"]
    )


def candidate_width_metrics(
    line_input: np.ndarray,
    edge_neighborhood: np.ndarray,
    endpoints: tuple[tuple[int, int], tuple[int, int]],
    high_response: float,
    chord_ratio: float,
) -> list[dict[str, float]]:
    primary = line_band_metrics(
        line_input,
        edge_neighborhood,
        endpoints,
        GLOBAL_CORE_WIDTH_RATIOS[0],
        high_response,
    )
    if primary is None:
        return []

    metrics = [primary]
    if validate_global_continuity(
        line_input, endpoints, high_response, chord_ratio, primary
    ):
        return metrics

    promising = bool(
        primary["bright_support"] >= GLOBAL_PROMISING_MIN_BRIGHT_SUPPORT
        and primary["edge_support"] >= GLOBAL_PROMISING_MIN_EDGE_SUPPORT
    )
    if not promising:
        return metrics

    for width_ratio in GLOBAL_CORE_WIDTH_RATIOS[1:]:
        item = line_band_metrics(
            line_input,
            edge_neighborhood,
            endpoints,
            width_ratio,
            high_response,
        )
        if item is None:
            continue
        metrics.append(item)
        if validate_global_continuity(
            line_input, endpoints, high_response, chord_ratio, item
        ):
            break
    return metrics


def detect_global_lines(
    line_input: np.ndarray,
    edges: np.ndarray,
    stop_after_first_clear: bool = True,
    include_bright_review: bool | None = None,
) -> tuple[int, list[dict[str, object]]]:
    if include_bright_review is None:
        include_bright_review = INCLUDE_BRIGHT_REVIEW
    height, width = edges.shape
    diagonal = math.hypot(width, height)
    high_response = float(np.percentile(line_input, 90))
    edge_neighborhood = cv2.dilate(
        edges,
        np.ones((5, 5), dtype=np.uint8),
        iterations=1,
    )
    candidates = []
    seen_peaks: list[tuple[float, float]] = []
    raw_peak_count = 0

    for peaks in global_peak_groups(
        line_input, edges, include_bright_review
    ):
        raw_peak_count += len(peaks)
        for source_name, rho, theta in peaks:
            duplicate_peak = False
            for seen_rho, seen_theta in seen_peaks:
                angle_delta = abs(theta - seen_theta)
                angle_delta = min(angle_delta, np.pi - angle_delta)
                if (
                    abs(rho - seen_rho) <= 3
                    and angle_delta <= np.deg2rad(2)
                ):
                    duplicate_peak = True
                    break
            if duplicate_peak:
                continue
            seen_peaks.append((rho, theta))

            endpoints = clipped_hough_line(rho, theta, width, height)
            if endpoints is None:
                continue
            (x1, y1), (x2, y2) = endpoints
            chord_ratio = math.hypot(x2 - x1, y2 - y1) / diagonal
            if chord_ratio < GLOBAL_REVIEW_MIN_CHORD_RATIO:
                continue

            width_metrics = candidate_width_metrics(
                line_input,
                edge_neighborhood,
                endpoints,
                high_response,
                chord_ratio,
            )
            if not width_metrics:
                continue
            classified = []
            for item in width_metrics:
                review, conservative, coherent = classify_global_metrics(
                    chord_ratio,
                    item,
                )
                classified.append((item, review, conservative, coherent))
            best_metrics, review_artifact, conservative_strict, coherent_strict = max(
                classified,
                key=lambda result: (
                    bool(result[2] or result[3]),
                    bool(result[1]),
                    metrics_quality(result[0]),
                ),
            )
            if source_name != GLOBAL_STRICT_SOURCE:
                conservative_strict = coherent_strict = False
            clear_artifact = conservative_strict or coherent_strict
            if not review_artifact and not clear_artifact:
                continue

            candidate = {
                "x1": float(x1),
                "y1": float(y1),
                "x2": float(x2),
                "y2": float(y2),
                "rho": rho,
                "theta_degrees": float(np.rad2deg(theta)),
                "source": source_name,
                "core_width": best_metrics["core_width"],
                "chord_ratio": float(chord_ratio),
                "bright_support": best_metrics["bright_support"],
                "contrast": best_metrics["contrast"],
                "edge_support": best_metrics["edge_support"],
                "supported_bright_bins": int(
                    best_metrics.get("supported_bright_bins", 0)
                ),
                "review_artifact": review_artifact,
                "clear_artifact": clear_artifact,
                "strict_rule": (
                    "conservative_and_coherent"
                    if conservative_strict and coherent_strict
                    else "conservative"
                    if conservative_strict
                    else "edge_coherent"
                    if coherent_strict
                    else "review"
                ),
            }
            candidate["score"] = global_line_artifact_score(candidate)
            candidates.append(candidate)
            if clear_artifact and stop_after_first_clear:
                return raw_peak_count, candidates

    return raw_peak_count, candidates


def line_artifact_score(segment: dict[str, float]) -> float:
    length_score = min(1.0, segment["length_ratio"] / 0.85)
    support_score = min(1.0, segment["bright_support"] / 0.60)
    contrast_score = min(
        1.0,
        max(0.0, segment["contrast"] - 1.0) / 0.40,
    )
    border_score = segment["border_endpoints"] / 2.0
    return float(
        0.35 * length_score
        + 0.30 * support_score
        + 0.20 * contrast_score
        + 0.15 * border_score
    )


def analyze_straight_lines(image_path: str) -> dict[str, object]:
    started = time.perf_counter()
    try:
        with Image.open(image_path) as image:
            rgb = np.asarray(image.convert("RGB"))
    except Exception as exc:
        return {
            "read_ok": False,
            "read_error": f"{type(exc).__name__}: {exc}",
            "read_seconds": time.perf_counter() - started,
            "compute_seconds": 0.0,
            "bright_review_enabled": bool(INCLUDE_BRIGHT_REVIEW),
        }

    read_finished = time.perf_counter()
    _, line_input, edges = preprocess_for_lines(rgb)
    raw_count, supported = detect_supported_lines(line_input, edges)
    clear = [segment for segment in supported if segment["clear_artifact"]]
    if clear:
        global_raw_count, global_supported = 0, []
    else:
        global_raw_count, global_supported = detect_global_lines(
            line_input,
            edges,
            stop_after_first_clear=True,
        )
    global_clear = [
        line for line in global_supported if line["clear_artifact"]
    ]
    strict_global = global_clear[0] if global_clear else None
    best = max(supported, key=line_artifact_score) if supported else None
    global_best = (
        max(global_supported, key=lambda line: line["score"])
        if global_supported else None
    )
    best_score = max(
        line_artifact_score(best) if best is not None else 0.0,
        float(global_best["score"]) if global_best is not None else 0.0,
    )

    return {
        "read_ok": True,
        "read_error": "",
        "read_seconds": read_finished - started,
        "compute_seconds": time.perf_counter() - read_finished,
        "bright_review_enabled": bool(INCLUDE_BRIGHT_REVIEW),
        "width": int(rgb.shape[1]),
        "height": int(rgb.shape[0]),
        "detection_size": DETECTION_SIZE,
        "raw_hough_line_count": float(raw_count),
        "supported_line_count": float(len(supported)),
        "probabilistic_clear_line_count": float(len(clear)),
        "global_hough_line_count": float(global_raw_count),
        "global_supported_line_count": float(len(global_supported)),
        "global_clear_line_count": float(len(global_clear)),
        "clear_line_count": float(len(clear) + len(global_clear)),
        "line_artifact_score": best_score,
        "max_line_length_ratio": (
            max(segment["length_ratio"] for segment in supported)
            if supported else 0.0
        ),
        "max_line_bright_support": (
            max(segment["bright_support"] for segment in supported)
            if supported else 0.0
        ),
        "max_line_contrast": (
            max(segment["contrast"] for segment in supported)
            if supported else 0.0
        ),
        "max_border_endpoint_count": (
            max(segment["border_endpoints"] for segment in supported)
            if supported else 0.0
        ),
        "max_global_chord_ratio": (
            max(line["chord_ratio"] for line in global_supported)
            if global_supported else 0.0
        ),
        "max_global_bright_support": (
            max(line["bright_support"] for line in global_supported)
            if global_supported else 0.0
        ),
        "max_global_contrast": (
            max(line["contrast"] for line in global_supported)
            if global_supported else 0.0
        ),
        "max_global_edge_support": (
            max(line["edge_support"] for line in global_supported)
            if global_supported else 0.0
        ),
        "best_global_core_width": (
            float(global_best["core_width"])
            if global_best is not None else 0.0
        ),
        "best_global_source": (
            str(global_best["source"])
            if global_best is not None else ""
        ),
        "best_global_strict_rule": (
            str(global_best["strict_rule"])
            if global_best is not None else ""
        ),
        "strict_global_source": (
            str(strict_global["source"]) if strict_global else ""
        ),
        "strict_global_rule": (
            str(strict_global["strict_rule"]) if strict_global else ""
        ),
        "strict_global_supported_bright_bins": (
            int(strict_global["supported_bright_bins"]) if strict_global else 0
        ),
        "clear_line_artifact": bool(clear or global_clear),
        "line_review_flag": bool(supported or global_supported),
    }

## Regression check for lines and visually line-free controls

The detector must recover both the original version 2 examples and the six
additional broad or fragmented artifacts found after clustering the filtered
catalogue, while retaining manually inspected line-free and edge-on cutouts. Run
this cell before launching the complete batch.

In [ ]:
KNOWN_STRAIGHT_LINE_ARTIFACTS = [
    # Original version 2 regression set.
    "102159780_2763153601669095508_gz_arcsinh_vis_only.jpg",
    "102158587_2720102675650853470_gz_arcsinh_vis_only.jpg",
    "102020058_NEG586092876494075221_gz_arcsinh_vis_only.jpg",
    "102018664_NEG568658583507586322_gz_arcsinh_vis_only.jpg",
    "102022474_NEG587678331471606201_gz_arcsinh_vis_only.jpg",
    # Additional artifacts recovered by version 3.
    "102158893_2726532238653121101_gz_arcsinh_vis_only.jpg",
    "102160063_2747044002674733150_gz_arcsinh_vis_only.jpg",
    "102021010_NEG583336158482801142_gz_arcsinh_vis_only.jpg",
    "102159483_2657612747667331984_gz_arcsinh_vis_only.jpg",
    "102159484_2665825185665237926_gz_arcsinh_vis_only.jpg",
    "102042286_NEG536384498295270675_gz_arcsinh_vis_only.jpg",
]

regression_rows = []
for filename in KNOWN_STRAIGHT_LINE_ARTIFACTS:
    tile = filename.split("_", 1)[0]
    image_path = CUTOUT_BASE / tile / filename
    result = analyze_straight_lines(str(image_path))
    regression_rows.append({
        "filename": filename,
        "read_ok": result.get("read_ok", False),
        "clear_line_artifact": result.get("clear_line_artifact", False),
        "global_clear_line_count": result.get("global_clear_line_count", 0),
        "strict_global_source": result.get("strict_global_source", ""),
        "best_global_core_width": result.get("best_global_core_width", 0),
        "strict_global_rule": result.get("strict_global_rule", ""),
        "strict_global_supported_bright_bins": result.get(
            "strict_global_supported_bright_bins", 0
        ),
        "line_artifact_score": result.get("line_artifact_score", 0),
        "read_error": result.get("read_error", ""),
    })

regression_df = pd.DataFrame(regression_rows)
display(regression_df)

failed_regressions = regression_df[
    ~regression_df["read_ok"] | ~regression_df["clear_line_artifact"]
]
if not failed_regressions.empty:
    raise AssertionError(
        "Known straight-line artifacts were missed; review v3.2 thresholds."
    )

VISUALLY_LINE_FREE_CONTROLS = [
    "102158893_2718217157653023054_gz_arcsinh_vis_only.jpg",
    "102019587_NEG567765110499908478_gz_arcsinh_vis_only.jpg",
    "102019128_NEG604577834504671728_gz_arcsinh_vis_only.jpg",
    "102159187_2646163055660697425_gz_arcsinh_vis_only.jpg",
    "102019594_NEG620859084498853033_gz_arcsinh_vis_only.jpg",
    "102019123_NEG570732944505032315_gz_arcsinh_vis_only.jpg",
    "102159191_2688593841658043186_gz_arcsinh_vis_only.jpg",
    "102157953_2665518947637936841_gz_arcsinh_vis_only.jpg",
    "102019127_NEG596643788507228667_gz_arcsinh_vis_only.jpg",
    "102157956_2696160515639449362_gz_arcsinh_vis_only.jpg",
    "102042915_NEG528179930292002288_gz_arcsinh_vis_only.jpg",
    "102159772_2661049275670107886_gz_arcsinh_vis_only.jpg",
]
negative_rows = []
for filename in VISUALLY_LINE_FREE_CONTROLS:
    tile = filename.split("_", 1)[0]
    result = analyze_straight_lines(str(CUTOUT_BASE / tile / filename))
    negative_rows.append({
        "filename": filename,
        "read_ok": result.get("read_ok", False),
        "clear_line_artifact": result.get("clear_line_artifact", False),
    })
negative_df = pd.DataFrame(negative_rows)
display(negative_df)
if (~negative_df["read_ok"] | negative_df["clear_line_artifact"]).any():
    raise AssertionError("A visually line-free control was excluded.")
print("Version 3.2 positive and negative regression checks passed.")

## Process in resumable batches

With `MAX_IMAGES = None`, this section processes the complete catalogue. Each
batch is written to a version-specific Parquet checkpoint, so reconnecting
Colab and rerunning the notebook skips object IDs that were already completed.
Use a different `DETECTOR_VERSION` whenever thresholds change. The optional
worker benchmark tests a small set of distinct pending images and picks the
fastest worker count for this Colab runtime. It does not change detections
or invalidate existing checkpoints. Per-batch read and compute timings help
separate Google Drive latency from detector CPU time; process/checkpoint
times show whether writing the Parquet parts is also expensive.

`INCLUDE_BRIGHT_REVIEW = False` skips only the residual Hough proposals,
which are review-only in v3.2; strict exclusions are unchanged. The review
manifest is therefore less exhaustive for newly processed rows, and the
`bright_review_enabled` column records this choice. Earlier checkpoints
are still valid for the exclusion catalogue. Set it to `True` only if a
complete review-only candidate list is needed.

If an older copy is currently processing, interrupt it and wait for the
running batch to stop before opening this copy. Rerun the setup and batch
cells with the same `DETECTOR_VERSION` and `OUTPUT_DIR`; completed Parquet
parts are reused. A `.parquet.tmp` file is not counted as completed and
its objects will be processed again. If read time dominates, compare the
worker benchmark results; if compute time dominates, more workers may not
help on a CPU-limited Colab runtime.

In [ ]:
from concurrent.futures import ThreadPoolExecutor
import time

from tqdm.auto import tqdm


def completed_object_ids() -> set[str]:
    completed = set()
    for path in sorted(PARTS_DIR.glob("straight_line_part_*.parquet")):
        part = pd.read_parquet(path, columns=["object_id"])
        completed.update(part["object_id"].astype(str))
    return completed


def process_row(row: dict) -> dict:
    return {**row, **analyze_straight_lines(row["image_path"])}


completed = completed_object_ids()
pending = manifest[~manifest["object_id"].isin(completed)].copy()
records = pending.to_dict("records")
part_numbers = [
    int(path.stem.rsplit("_", 1)[-1])
    for path in PARTS_DIR.glob("straight_line_part_*.parquet")
]
next_part = max(part_numbers, default=-1) + 1
print(f"Already completed: {len(completed):,}")
print(f"Pending: {len(pending):,}")

effective_workers = N_WORKERS
worker_options = tuple(sorted(set(WORKER_OPTIONS)))
if AUTO_TUNE_WORKERS and worker_options and records:
    sample_size = min(
        WORKER_BENCHMARK_IMAGES, len(records) // len(worker_options)
    )
    if sample_size >= 16:
        rng = np.random.default_rng(RANDOM_SEED)
        indices = rng.choice(
            len(records), size=sample_size * len(worker_options),
            replace=False,
        )
        timings = {}
        for option_index, workers in enumerate(worker_options):
            offset = option_index * sample_size
            sample = [records[int(index)] for index in (
                indices[offset:offset + sample_size]
            )]
            benchmark_started = time.perf_counter()
            with ThreadPoolExecutor(max_workers=workers) as executor:
                list(executor.map(process_row, sample))
            timings[workers] = (
                time.perf_counter() - benchmark_started
            ) / sample_size
            print(f"{workers} workers: {timings[workers]:.3f} s/image")
        effective_workers = min(timings, key=timings.get)
        print(f"Selected {effective_workers} workers for this runtime.")
    else:
        print("Too few pending images to tune workers; using fallback.")

run_started = time.perf_counter()

for start in range(0, len(records), BATCH_SIZE):
    batch = records[start:start + BATCH_SIZE]
    batch_started = time.perf_counter()
    with ThreadPoolExecutor(max_workers=effective_workers) as executor:
        results = list(tqdm(
            executor.map(process_row, batch),
            total=len(batch),
            desc=f"Batch {next_part:05d}",
        ))
    processing_seconds = time.perf_counter() - batch_started
    part_path = PARTS_DIR / f"straight_line_part_{next_part:05d}.parquet"
    temporary_path = part_path.with_suffix(".parquet.tmp")
    pd.DataFrame(results).to_parquet(temporary_path, index=False)
    temporary_path.replace(part_path)
    checkpoint_seconds = time.perf_counter() - batch_started - processing_seconds

    processed_this_run = min(start + len(batch), len(records))
    remaining = len(records) - processed_this_run
    elapsed = time.perf_counter() - run_started
    rate = processed_this_run / elapsed if elapsed else 0.0
    eta_hours = remaining / rate / 3600 if rate else float("inf")
    batch_minutes = (time.perf_counter() - batch_started) / 60
    successful = [row for row in results if row.get("read_ok")]
    read_ms = (
        1_000 * np.median([row["read_seconds"] for row in successful])
        if successful else float("nan")
    )
    compute_ms = (
        1_000 * np.median([row["compute_seconds"] for row in successful])
        if successful else float("nan")
    )
    print(
        f"Batch {next_part:05d}: {batch_minutes:.2f} min | "
        f"processed this run: {processed_this_run:,}/{len(records):,} | "
        f"estimated remaining: {eta_hours:.1f} h | "
        f"median read/compute: {read_ms:.0f}/{compute_ms:.0f} ms | "
        f"process/checkpoint: {processing_seconds:.1f}/"
        f"{checkpoint_seconds:.1f} s"
    )
    next_part += 1

print("Batch processing complete.")

## Consolidate results

A supported line that is not sufficiently border-to-border is sent to review,
not exclusion. Labelled candidates remain protected even when a clear line is
detected.


In [ ]:
part_paths = sorted(PARTS_DIR.glob("straight_line_part_*.parquet"))
if not part_paths:
    raise RuntimeError("No straight-line checkpoint files were found.")

metrics_df = pd.concat(
    [pd.read_parquet(path) for path in part_paths],
    ignore_index=True,
).drop_duplicates("object_id", keep="last")
if "bright_review_enabled" not in metrics_df:
    metrics_df["bright_review_enabled"] = True
else:
    metrics_df["bright_review_enabled"] = (
        metrics_df["bright_review_enabled"].fillna(True).astype(bool)
    )
expected_ids = set(manifest["object_id"].astype(str))
processed_ids = set(metrics_df["object_id"].astype(str))
missing_ids = expected_ids - processed_ids
unexpected_ids = processed_ids - expected_ids
if missing_ids or unexpected_ids:
    raise RuntimeError(
        f"Incomplete batch results: {len(missing_ids)} missing, "
        f"{len(unexpected_ids)} unexpected object IDs."
    )

metrics_df["review_flag"] = (
    ~metrics_df["read_ok"]
    | metrics_df["line_review_flag"]
)
metrics_df["exclude_straight_line"] = (
    metrics_df["read_ok"]
    & metrics_df["clear_line_artifact"]
    & ~metrics_df["is_lens_candidate"]
)

metrics_path = OUTPUT_DIR / f"straight_line_metrics_{DETECTOR_VERSION}.parquet"
review_path = OUTPUT_DIR / f"straight_line_review_{DETECTOR_VERSION}.csv"
exclusion_path = OUTPUT_DIR / f"straight_line_exclusions_{DETECTOR_VERSION}.csv"

diagnostic_columns = [
    "object_id",
    "id_str",
    "grade",
    "image_path",
    "line_artifact_score",
    "raw_hough_line_count",
    "global_hough_line_count",
    "supported_line_count",
    "global_supported_line_count",
    "clear_line_count",
    "probabilistic_clear_line_count",
    "global_clear_line_count",
    "max_line_length_ratio",
    "max_line_bright_support",
    "max_line_contrast",
    "max_global_chord_ratio",
    "max_global_bright_support",
    "max_global_contrast",
    "max_global_edge_support",
    "best_global_source",
    "best_global_core_width",
    "best_global_strict_rule",
    "strict_global_source",
    "strict_global_rule",
    "strict_global_supported_bright_bins",
    "bright_review_enabled",
    "read_ok",
    "read_error",
]

metrics_df.to_parquet(metrics_path, index=False)
metrics_df.loc[
    metrics_df["review_flag"],
    diagnostic_columns,
].sort_values("line_artifact_score", ascending=False).to_csv(
    review_path,
    index=False,
)
metrics_df.loc[
    metrics_df["exclude_straight_line"],
    diagnostic_columns,
].sort_values("line_artifact_score", ascending=False).to_csv(
    exclusion_path,
    index=False,
)

print(f"Metrics: {metrics_path}")
print(f"Review manifest: {review_path}")
print(f"Strict exclusion manifest: {exclusion_path}")
print(f"Review objects: {int(metrics_df['review_flag'].sum()):,}")
print(f"Strict exclusions: {int(metrics_df['exclude_straight_line'].sum()):,}")
print(
    "Protected labelled candidates among strict detections: "
    f"{int((metrics_df['clear_line_artifact'] & metrics_df['is_lens_candidate']).sum()):,}"
)

## Diagnostic gallery

Red lines satisfy the strict artifact rule. Orange lines pass support and
contrast checks but remain review-only. Raw Hough proposals that fail those
checks are not drawn.


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle


def line_overlay(row) -> np.ndarray:
    with Image.open(row.image_path) as image:
        rgb = np.asarray(image.convert("RGB")).copy()
    analysis_rgb, line_input, edges = preprocess_for_lines(rgb)
    _, supported = detect_supported_lines(line_input, edges)
    _, global_supported = detect_global_lines(
        line_input, edges, include_bright_review=True
    )

    for segment in supported:
        color = (255, 40, 40) if segment["clear_artifact"] else (255, 170, 0)
        cv2.line(
            analysis_rgb,
            (int(segment["x1"]), int(segment["y1"])),
            (int(segment["x2"]), int(segment["y2"])),
            color,
            2,
        )
    for line in global_supported:
        color = (255, 40, 40) if line["clear_artifact"] else (0, 210, 255)
        cv2.line(
            analysis_rgb,
            (int(line["x1"]), int(line["y1"])),
            (int(line["x2"]), int(line["y2"])),
            color,
            max(2, int(round(line["core_width"]))),
        )
    return analysis_rgb


def show_gallery(rows: pd.DataFrame, title: str, columns: int = 5) -> None:
    rows = rows.reset_index(drop=True)
    if rows.empty:
        print(f"{title}: no objects")
        return

    n_rows = math.ceil(len(rows) / columns)
    figure, axes = plt.subplots(
        n_rows,
        columns,
        figsize=(3.2 * columns, 3.6 * n_rows),
    )
    axes = np.atleast_1d(axes).ravel()
    for axis, row in zip(axes, rows.itertuples(index=False)):
        try:
            axis.imshow(line_overlay(row))
        except Exception:
            axis.text(0.5, 0.5, "Read error", ha="center", va="center")
        status = "EXCLUDE" if row.exclude_straight_line else "REVIEW"
        axis.set_title(
            f"{status} | score={row.line_artifact_score:.2f}\n"
            f"{row.object_id}",
            fontsize=8,
        )
        frame_color = "red" if row.exclude_straight_line else "orange"
        axis.add_patch(Rectangle(
            (0, 0),
            1,
            1,
            transform=axis.transAxes,
            fill=False,
            edgecolor=frame_color,
            linewidth=3,
        ))
        axis.axis("off")
    for axis in axes[len(rows):]:
        axis.axis("off")
    figure.suptitle(title, fontsize=14)
    plt.tight_layout()
    plt.show()


suspects = (
    metrics_df[metrics_df["review_flag"] & metrics_df["read_ok"]]
    .sort_values("line_artifact_score", ascending=False)
    .head(30)
)
show_gallery(suspects, "Supported straight-line detections")

In [ ]:
controls = metrics_df[
    metrics_df["read_ok"] & ~metrics_df["line_review_flag"]
]
if len(controls):
    controls = controls.sample(
        n=min(20, len(controls)),
        random_state=RANDOM_SEED,
    )
    show_gallery(controls, "Random controls without supported lines")

strict_sample = metrics_df[metrics_df["exclude_straight_line"]]
if len(strict_sample):
    strict_sample = strict_sample.sample(
        n=min(30, len(strict_sample)), random_state=RANDOM_SEED
    )
    show_gallery(strict_sample, "Random automatic exclusions")

review_sample = metrics_df[
    metrics_df["read_ok"]
    & metrics_df["review_flag"]
    & ~metrics_df["exclude_straight_line"]
]
if len(review_sample):
    review_sample = review_sample.sample(
        n=min(30, len(review_sample)), random_state=RANDOM_SEED
    )
    show_gallery(review_sample, "Random review-only candidates")

## Optional filtered PCA catalogue for ESCOPE

This creates a new file and preserves the original PCA catalogue. Only strict
straight-line exclusions are removed. Review-only candidates stay in the
catalogue. This cell refuses to publish a sample run as a full filter.


In [ ]:
if MAX_IMAGES is not None:
    raise RuntimeError("Run the complete catalogue before exporting a PCA filter.")

PARQUET_PATH = Path(
    "/content/drive/MyDrive/catalogues/morphology_catalogue/"
    "representations_pca_40.parquet"
)
FILTERED_PARQUET_PATH = (
    OUTPUT_DIR
    / f"representations_pca_40_artifacts_filtered_{DETECTOR_VERSION}.parquet"
)

excluded_ids = set(metrics_df.loc[
    metrics_df["exclude_straight_line"],
    "object_id",
].astype(str))

pca_df = pd.read_parquet(PARQUET_PATH)
if "object_id" not in pca_df.columns:
    pca_df["object_id"] = pca_df["id_str"].astype(str).str.split("_").str[-1]
normalized_ids = pca_df["object_id"].map(normalize_object_id)
filtered_pca_df = pca_df.loc[~normalized_ids.isin(excluded_ids)].copy()
filtered_pca_df.to_parquet(FILTERED_PARQUET_PATH, index=False)

print(f"Original PCA rows: {len(pca_df):,}")
print(f"Excluded rows: {len(pca_df) - len(filtered_pca_df):,}")
print(f"Filtered PCA rows: {len(filtered_pca_df):,}")
print(f"Written to: {FILTERED_PARQUET_PATH}")

## Validation before using version 3.2 in ESCOPE

1. Run the 11-positive and 12-negative regression checks first.
2. For threshold tuning, set `MAX_IMAGES` to a random sample and use a new
   `DETECTOR_VERSION`; checkpoint parts are version-specific.
3. Review random automatic exclusions and review-only candidates, including
   edge-on galaxies and protected A/B/C candidates. A high score alone is not
   evidence of an instrumental artifact.
4. Run with `MAX_IMAGES = None`; the manifest cell must report zero missing
   object IDs. Inspect read errors before exporting the filtered PCA file.
5. Compare exclusion counts and manually reviewed samples against v2 and
   v3.1. Publish the new parquet to Cloud Storage only after visual QA.
6. Keep previous parquets and record this detector version, thresholds, and
   review protocol with any scientific result.

Red overlays are strict exclusions, orange overlays are probabilistic review
candidates, and cyan overlays are global-Hough review candidates. The source
catalogues and images are never modified.